# Weekday/Weekend XGBoost Ensemble with MAD Robustness

This notebook trains:

1. A global XGBoost model on all rows.
2. A weekday specialist XGBoost model.
3. A weekend specialist XGBoost model.
4. A validation-tuned ensemble that blends the global model with the correct specialist.

It also adds robust time-series features, applies MAD-based outlier handling, tunes hyperparameters, reports validation metrics, and writes `submission.csv` for Kaggle.

**Kaggle:** set Notebook Accelerator to GPU. The code requests CUDA for XGBoost and falls back to CPU only if GPU training is unavailable.


In [32]:
# =====================
# Configuration
# =====================
from pathlib import Path
import os
import math
import warnings
warnings.filterwarnings("ignore")

RANDOM_STATE = 42
TARGET = "consumption"
DATE_COL = "start_time"

# Use a smaller number while testing, then increase for a final run.
TUNE_MODELS = True
N_TRIALS_GLOBAL = 30
N_TRIALS_SPECIALIST = 25
VALID_FRACTION = 0.20

# MAD robustness controls.
MAD_Z_TARGET = 6.0
MAD_Z_FEATURES = 8.0
USE_MAD_TARGET_CAP = True
USE_MAD_SAMPLE_WEIGHTS = True
USE_MAD_FEATURE_CLIP = True

# XGBoost runtime.
EARLY_STOPPING_ROUNDS = 100
DEFAULT_N_ESTIMATORS = 3000
USE_GPU = True
GPU_DEVICE = "cuda"
CPU_TREE_METHOD = "hist"
N_JOBS = -1

# Local fallback paths. On Kaggle, the notebook auto-discovers these files under /kaggle/input.
LOCAL_TRAIN_PATH = Path(r"C:\Users\Chavda\Desktop\Hackathon\train_clean.csv")
LOCAL_TEST_PATH = Path(r"C:\Users\Chavda\Desktop\Hackathon\test_clean.csv")
OUTPUT_DIR = Path("/kaggle/working") if Path("/kaggle").exists() else Path(".")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


In [33]:
# =====================
# Imports
# =====================
import numpy as np
import pandas as pd

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import ParameterSampler

try:
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    HAS_OPTUNA = True
except Exception:
    HAS_OPTUNA = False

import xgboost as xgb
from xgboost import XGBRegressor

print(f"Optuna available: {HAS_OPTUNA}")
print(f"XGBoost version: {xgb.__version__}")
print(f"GPU requested: {USE_GPU}")


Optuna available: True
XGBoost version: 3.2.0
GPU requested: True


In [34]:
# =====================
# File discovery
# =====================
def find_kaggle_file(filename):
    root = Path("/kaggle/input")
    if not root.exists():
        return None
    matches = sorted(root.rglob(filename))
    return matches[0] if matches else None


def resolve_data_paths():
    kaggle_train = find_kaggle_file("train_clean.csv")
    kaggle_test = find_kaggle_file("test_clean.csv")

    train_path = kaggle_train if kaggle_train is not None else LOCAL_TRAIN_PATH
    test_path = kaggle_test if kaggle_test is not None else LOCAL_TEST_PATH

    if not train_path.exists():
        raise FileNotFoundError(f"Train file not found: {train_path}")
    if not test_path.exists():
        raise FileNotFoundError(f"Test file not found: {test_path}")

    return train_path, test_path


TRAIN_PATH, TEST_PATH = resolve_data_paths()
print("Train:", TRAIN_PATH)
print("Test :", TEST_PATH)


Train: /kaggle/input/datasets/vivekk777/nnjnjnj/train_clean.csv
Test : /kaggle/input/datasets/vivekk777/nnjnjnj/test_clean.csv


In [35]:
# =====================
# Load data
# =====================
train_raw = pd.read_csv(TRAIN_PATH)
test_raw = pd.read_csv(TEST_PATH)

for df in (train_raw, test_raw):
    if DATE_COL in df.columns:
        df[DATE_COL] = pd.to_datetime(df[DATE_COL], errors="coerce")

print("Train shape:", train_raw.shape)
print("Test shape :", test_raw.shape)
display(train_raw.head())


Train shape: (43515, 21)
Test shape : (8757, 21)


,start_time,consumption,hour,day_of_week,month,year,season,is_weekend,is_holiday,is_first_day,...,lag_24h,lag_48h,lag_72h,lag_168h,prev_daily_mean,prev_daily_max,rolling_mean_24h,rolling_mean_168h,rolling_std_24h,ewm_24h
0,2016-01-14 21:00:00,12591.0,21.0,3.0,1.0,2016.0,0.0,0.0,0.0,0.0,...,12597.0,12385.0,12378.0,14074.0,13283.541667,13631.0,13202.666667,13210.854167,229.903887,13222.077680
1,2016-01-14 22:00:00,13155.0,22.0,3.0,1.0,2016.0,0.0,0.0,0.0,0.0,...,13225.0,13076.0,13035.0,13643.0,13283.541667,13631.0,13202.416667,13202.026786,230.593359,13171.591466
2,2016-01-14 23:00:00,13216.5,23.0,3.0,1.0,2016.0,0.0,0.0,0.0,0.0,...,13193.0,13081.5,13035.0,14639.0,13283.541667,13631.0,13199.500000,13199.122024,230.737947,13170.264149
3,2016-01-15 00:00:00,13290.5,0.0,4.0,1.0,2016.0,0.0,0.0,0.0,0.0,...,13178.0,13124.5,13035.0,14592.0,13200.479167,13567.0,13200.479167,13190.654762,230.759025,13173.963017
4,2016-01-15 01:00:00,13290.5,1.0,4.0,1.0,2016.0,0.0,0.0,0.0,0.0,...,13150.0,13196.5,13020.5,14483.5,13200.479167,13567.0,13205.166667,13182.907738,231.424214,13183.285976


In [36]:
# =====================
# Metrics and helpers
# =====================
def rmse(y_true, y_pred):
    try:
        return mean_squared_error(y_true, y_pred, squared=False)
    except TypeError:
        return float(np.sqrt(mean_squared_error(y_true, y_pred)))


def mape(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    denom = np.where(np.abs(y_true) < 1e-9, np.nan, np.abs(y_true))
    return np.nanmean(np.abs((y_true - y_pred) / denom)) * 100


def smape(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    denom = (np.abs(y_true) + np.abs(y_pred)) / 2
    denom = np.where(denom < 1e-9, np.nan, denom)
    return np.nanmean(np.abs(y_true - y_pred) / denom) * 100


def metric_row(name, y_true, y_pred):
    return {
        "model": name,
        "rmse": rmse(y_true, y_pred),
        "mae": mean_absolute_error(y_true, y_pred),
        "mape_pct": mape(y_true, y_pred),
        "smape_pct": smape(y_true, y_pred),
        "r2": r2_score(y_true, y_pred),
    }


def show_metrics(rows):
    table = pd.DataFrame(rows).sort_values("rmse").reset_index(drop=True)
    display(table)
    return table


def safe_divide(a, b):
    return np.where(np.abs(b) < 1e-9, np.nan, a / b)


In [37]:
# =====================
# Feature engineering
# =====================
def add_base_features(df):
    df = df.copy()

    if DATE_COL in df.columns:
        dt = pd.to_datetime(df[DATE_COL], errors="coerce")
        if "hour" not in df.columns:
            df["hour"] = dt.dt.hour
        if "day_of_week" not in df.columns:
            df["day_of_week"] = dt.dt.dayofweek
        if "month" not in df.columns:
            df["month"] = dt.dt.month
        if "year" not in df.columns:
            df["year"] = dt.dt.year
        df["day_of_month"] = dt.dt.day
        df["day_of_year"] = dt.dt.dayofyear
        df["week_of_year"] = dt.dt.isocalendar().week.astype(float)
        df["quarter"] = dt.dt.quarter

    for col in ["hour", "day_of_week", "month", "year"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    if "is_weekend" not in df.columns and "day_of_week" in df.columns:
        df["is_weekend"] = df["day_of_week"].isin([5, 6]).astype(float)

    df["hour_of_week"] = df["day_of_week"] * 24 + df["hour"]
    df["is_business_hour"] = ((df["hour"].between(8, 18)) & (df["is_weekend"] == 0)).astype(float)
    df["is_night"] = df["hour"].isin([0, 1, 2, 3, 4, 5]).astype(float)
    df["is_morning_peak"] = df["hour"].between(7, 10).astype(float)
    df["is_evening_peak"] = df["hour"].between(17, 22).astype(float)
    df["is_holiday_or_weekend"] = ((df.get("is_holiday", 0) == 1) | (df["is_weekend"] == 1)).astype(float)

    # Cyclical encodings.
    df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
    df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)
    df["dow_sin"] = np.sin(2 * np.pi * df["day_of_week"] / 7)
    df["dow_cos"] = np.cos(2 * np.pi * df["day_of_week"] / 7)
    df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12)
    df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12)
    df["hour_of_week_sin"] = np.sin(2 * np.pi * df["hour_of_week"] / 168)
    df["hour_of_week_cos"] = np.cos(2 * np.pi * df["hour_of_week"] / 168)

    # Lag/rolling interactions.
    lag_cols = [c for c in ["lag_24h", "lag_48h", "lag_72h", "lag_168h"] if c in df.columns]
    for col in lag_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    if {"lag_24h", "lag_168h"}.issubset(df.columns):
        df["lag_diff_24_168"] = df["lag_24h"] - df["lag_168h"]
        df["lag_ratio_24_168"] = safe_divide(df["lag_24h"], df["lag_168h"])
        df["lag_abs_diff_24_168"] = np.abs(df["lag_diff_24_168"])

    if {"lag_24h", "lag_48h", "lag_72h"}.issubset(df.columns):
        df["lag_mean_24_72"] = df[["lag_24h", "lag_48h", "lag_72h"]].mean(axis=1)
        df["lag_std_24_72"] = df[["lag_24h", "lag_48h", "lag_72h"]].std(axis=1)
        df["lag_trend_24_72"] = df["lag_24h"] - df["lag_72h"]

    if {"rolling_mean_24h", "rolling_mean_168h"}.issubset(df.columns):
        df["rolling_diff_24_168"] = df["rolling_mean_24h"] - df["rolling_mean_168h"]
        df["rolling_ratio_24_168"] = safe_divide(df["rolling_mean_24h"], df["rolling_mean_168h"])

    if {"rolling_std_24h", "rolling_mean_24h"}.issubset(df.columns):
        df["rolling_cv_24h"] = safe_divide(df["rolling_std_24h"], df["rolling_mean_24h"])

    if {"prev_daily_max", "prev_daily_mean"}.issubset(df.columns):
        df["daily_peak_gap"] = df["prev_daily_max"] - df["prev_daily_mean"]
        df["daily_peak_ratio"] = safe_divide(df["prev_daily_max"], df["prev_daily_mean"])

    if {"ewm_24h", "rolling_mean_24h"}.issubset(df.columns):
        df["ewm_minus_roll24"] = df["ewm_24h"] - df["rolling_mean_24h"]

    df = df.replace([np.inf, -np.inf], np.nan)
    return df


def add_target_stat_features(reference_df, frames, target=TARGET):
    """Fit target aggregates on reference_df only, then add them to every frame."""
    out_frames = [frame.copy() for frame in frames]
    global_mean = reference_df[target].mean()
    global_median = reference_df[target].median()

    group_specs = [
        (["hour_of_week"], "how"),
        (["hour"], "hour"),
        (["day_of_week"], "dow"),
        (["month"], "month"),
        (["is_weekend", "hour"], "weekend_hour"),
    ]

    for group_cols, prefix in group_specs:
        available = [c for c in group_cols if c in reference_df.columns]
        if len(available) != len(group_cols):
            continue

        stats = (
            reference_df.groupby(group_cols)[target]
            .agg(["mean", "median", "std"])
            .reset_index()
            .rename(columns={
                "mean": f"{prefix}_target_mean",
                "median": f"{prefix}_target_median",
                "std": f"{prefix}_target_std",
            })
        )

        new_frames = []
        for frame in out_frames:
            merged = frame.merge(stats, on=group_cols, how="left")
            for col in [f"{prefix}_target_mean", f"{prefix}_target_median", f"{prefix}_target_std"]:
                if col.endswith("_mean"):
                    merged[col] = merged[col].fillna(global_mean)
                elif col.endswith("_median"):
                    merged[col] = merged[col].fillna(global_median)
                else:
                    merged[col] = merged[col].fillna(0)
            new_frames.append(merged)
        out_frames = new_frames

    return out_frames


In [38]:
# =====================
# MAD robustness
# =====================
def mad_stats(values):
    values = pd.Series(values).astype(float)
    med = values.median()
    mad = np.median(np.abs(values - med))
    if not np.isfinite(mad) or mad < 1e-9:
        std = values.std()
        mad = std * 0.6745 if np.isfinite(std) and std > 1e-9 else 1.0
    return med, mad


def mad_cap_target(df, target=TARGET, group_col="hour_of_week", z_thresh=MAD_Z_TARGET):
    temp = df[[group_col, target]].copy()
    global_med, global_mad = mad_stats(temp[target])

    rows = []
    for key, grp in temp.groupby(group_col):
        med, mad = mad_stats(grp[target])
        rows.append((key, med, mad))

    stat = pd.DataFrame(rows, columns=[group_col, "_target_med", "_target_mad"])
    temp = temp.merge(stat, on=group_col, how="left")
    temp["_target_med"] = temp["_target_med"].fillna(global_med)
    temp["_target_mad"] = temp["_target_mad"].fillna(global_mad).clip(lower=1e-9)

    robust_z = 0.6745 * (temp[target] - temp["_target_med"]) / temp["_target_mad"]
    cap_delta = z_thresh * temp["_target_mad"] / 0.6745
    capped = temp["_target_med"] + np.clip(temp[target] - temp["_target_med"], -cap_delta, cap_delta)

    weights = np.ones(len(temp), dtype=float)
    if USE_MAD_SAMPLE_WEIGHTS:
        abs_z = np.abs(robust_z.fillna(0).to_numpy())
        weights = np.where(abs_z <= z_thresh, 1.0, np.maximum(0.25, z_thresh / np.maximum(abs_z, 1e-9)))

    info = {
        "outlier_rate": float((np.abs(robust_z) > z_thresh).mean()),
        "max_abs_robust_z": float(np.nanmax(np.abs(robust_z))),
    }

    if USE_MAD_TARGET_CAP:
        return capped.to_numpy(dtype=float), weights, info
    return temp[target].to_numpy(dtype=float), weights, info


class MADFeatureClipper:
    def __init__(self, z_thresh=MAD_Z_FEATURES):
        self.z_thresh = z_thresh
        self.bounds_ = {}

    def fit(self, X):
        self.bounds_ = {}
        for col in X.columns:
            s = pd.to_numeric(X[col], errors="coerce")
            if s.notna().sum() == 0:
                continue
            med, mad = mad_stats(s.dropna())
            delta = self.z_thresh * mad / 0.6745
            self.bounds_[col] = (med - delta, med + delta)
        return self

    def transform(self, X):
        X = X.copy()
        for col, (lo, hi) in self.bounds_.items():
            if col in X.columns:
                X[col] = pd.to_numeric(X[col], errors="coerce").clip(lo, hi)
        return X


In [39]:
# =====================
# Validation split
# =====================
train_base = add_base_features(train_raw)
test_base = add_base_features(test_raw)

if TARGET not in train_base.columns:
    raise ValueError(f"Missing target column: {TARGET}")

sort_cols = [DATE_COL] if DATE_COL in train_base.columns else None
if sort_cols:
    train_base = train_base.sort_values(DATE_COL).reset_index(drop=True)

cut = int(len(train_base) * (1 - VALID_FRACTION))
train_part_base = train_base.iloc[:cut].reset_index(drop=True)
valid_part_base = train_base.iloc[cut:].reset_index(drop=True)

# Target-stat features are fit only on train_part for validation to avoid leakage.
train_part_fe, valid_part_fe = add_target_stat_features(
    train_part_base,
    [train_part_base, valid_part_base],
    target=TARGET,
)

print("Train part:", train_part_fe.shape)
print("Valid part:", valid_part_fe.shape)
print("Validation dates:", valid_part_fe[DATE_COL].min(), "to", valid_part_fe[DATE_COL].max())
print("Weekend share train:", train_part_fe["is_weekend"].mean())
print("Weekend share valid:", valid_part_fe["is_weekend"].mean())


Train part: (34812, 66)
Valid part: (8703, 66)
Validation dates: 2020-01-04 09:00:00 to 2020-12-31 23:00:00
Weekend share train: 0.2856773526370217
Weekend share valid: 0.28576352981730435


In [40]:
# =====================
# Matrix preparation
# =====================
def get_feature_columns(df):
    blocked = {TARGET, DATE_COL}
    cols = [c for c in df.columns if c not in blocked]
    numeric_cols = []
    for c in cols:
        if pd.api.types.is_numeric_dtype(df[c]):
            numeric_cols.append(c)
    return numeric_cols


FEATURES = get_feature_columns(train_part_fe)
print(f"Feature count: {len(FEATURES)}")
print(FEATURES)


def make_xy(df, feature_cols=FEATURES, target=TARGET):
    X = df[feature_cols].copy()
    X = X.replace([np.inf, -np.inf], np.nan)
    y = df[target].astype(float).to_numpy() if target in df.columns else None
    return X, y


X_train, y_train_raw = make_xy(train_part_fe)
X_valid, y_valid = make_xy(valid_part_fe)

if USE_MAD_FEATURE_CLIP:
    feature_clipper = MADFeatureClipper(MAD_Z_FEATURES).fit(X_train)
    X_train = feature_clipper.transform(X_train)
    X_valid = feature_clipper.transform(X_valid)

y_train, global_weights, global_mad_info = mad_cap_target(train_part_fe, TARGET, "hour_of_week", MAD_Z_TARGET)
print("Global MAD info:", global_mad_info)


Feature count: 64
['hour', 'day_of_week', 'month', 'year', 'season', 'is_weekend', 'is_holiday', 'is_first_day', 'is_last_day', 'lag_24h', 'lag_48h', 'lag_72h', 'lag_168h', 'prev_daily_mean', 'prev_daily_max', 'rolling_mean_24h', 'rolling_mean_168h', 'rolling_std_24h', 'ewm_24h', 'day_of_month', 'day_of_year', 'week_of_year', 'quarter', 'hour_of_week', 'is_business_hour', 'is_night', 'is_morning_peak', 'is_evening_peak', 'is_holiday_or_weekend', 'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'month_sin', 'month_cos', 'hour_of_week_sin', 'hour_of_week_cos', 'lag_diff_24_168', 'lag_ratio_24_168', 'lag_abs_diff_24_168', 'lag_mean_24_72', 'lag_std_24_72', 'lag_trend_24_72', 'rolling_diff_24_168', 'rolling_ratio_24_168', 'rolling_cv_24h', 'daily_peak_gap', 'daily_peak_ratio', 'ewm_minus_roll24', 'how_target_mean', 'how_target_median', 'how_target_std', 'hour_target_mean', 'hour_target_median', 'hour_target_std', 'dow_target_mean', 'dow_target_median', 'dow_target_std', 'month_target_mean', '

In [41]:
# =====================
# XGBoost tuning
# =====================
def xgb_major_version():
    try:
        return int(str(xgb.__version__).split(".")[0])
    except Exception:
        return 2


def build_base_xgb_params(use_gpu=USE_GPU):
    params = {
        "objective": "reg:squarederror",
        "eval_metric": "rmse",
        "tree_method": CPU_TREE_METHOD,
        "random_state": RANDOM_STATE,
        "n_jobs": N_JOBS,
        "n_estimators": DEFAULT_N_ESTIMATORS,
    }
    if use_gpu:
        if xgb_major_version() >= 2:
            # XGBoost 2.x/3.x GPU syntax.
            params.update({"tree_method": "hist", "device": GPU_DEVICE})
        else:
            # Older Kaggle images used this syntax.
            params.update({"tree_method": "gpu_hist", "predictor": "gpu_predictor"})
    return params


BASE_XGB_PARAMS = build_base_xgb_params(USE_GPU)
CPU_XGB_PARAMS = build_base_xgb_params(False)
print("XGBoost training params:", {k: BASE_XGB_PARAMS[k] for k in BASE_XGB_PARAMS if k in ["tree_method", "device", "predictor"]})


def make_model(params=None, n_estimators=None, early_stopping=True):
    merged = dict(BASE_XGB_PARAMS)
    if params:
        merged.update(params)
    if n_estimators is not None:
        merged["n_estimators"] = int(n_estimators)
    if early_stopping:
        merged["early_stopping_rounds"] = EARLY_STOPPING_ROUNDS
    return XGBRegressor(**merged)


GPU_FALLBACK_MARKERS = [
    "cuda", "gpu", "no visible gpu", "must have at least one device", "deviceordinal",
    "all kernels tried for training failed", "gpu_hist", "cudart",
]


def looks_like_gpu_error(error):
    text = str(error).lower()
    return any(marker in text for marker in GPU_FALLBACK_MARKERS)


def move_model_to_cpu(model):
    params = model.get_params()
    updates = {"tree_method": CPU_TREE_METHOD}
    if params.get("device", None) is not None:
        updates["device"] = "cpu"
    if params.get("predictor", None) is not None:
        updates["predictor"] = "auto"
    model.set_params(**updates)
    return model


def _fit_model_once(model, X_tr, y_tr, X_va=None, y_va=None, sample_weight=None):
    if X_va is not None and y_va is not None:
        try:
            model.fit(
                X_tr,
                y_tr,
                sample_weight=sample_weight,
                eval_set=[(X_va, y_va)],
                verbose=False,
            )
        except TypeError:
            model.set_params(early_stopping_rounds=None)
            model.fit(
                X_tr,
                y_tr,
                sample_weight=sample_weight,
                eval_set=[(X_va, y_va)],
                verbose=False,
                early_stopping_rounds=EARLY_STOPPING_ROUNDS,
            )
    else:
        model.fit(X_tr, y_tr, sample_weight=sample_weight, verbose=False)
    return model


def fit_model(model, X_tr, y_tr, X_va=None, y_va=None, sample_weight=None):
    try:
        return _fit_model_once(model, X_tr, y_tr, X_va, y_va, sample_weight)
    except Exception as exc:
        if USE_GPU and looks_like_gpu_error(exc):
            print(f"GPU training failed, retrying on CPU. Reason: {str(exc)[:180]}")
            model = move_model_to_cpu(model)
            return _fit_model_once(model, X_tr, y_tr, X_va, y_va, sample_weight)
        raise


def best_rounds(model, fallback=800):
    for attr in ["best_iteration", "best_ntree_limit"]:
        value = getattr(model, attr, None)
        if value is not None:
            try:
                value = int(value)
                return max(50, value + (1 if attr == "best_iteration" else 0))
            except Exception:
                pass
    return fallback


def optuna_params(trial):
    return {
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.08, log=True),
        "subsample": trial.suggest_float("subsample", 0.65, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.65, 1.0),
        "min_child_weight": trial.suggest_float("min_child_weight", 1.0, 20.0, log=True),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-4, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 0.1, 30.0, log=True),
        "gamma": trial.suggest_float("gamma", 0.0, 5.0),
        "max_bin": trial.suggest_int("max_bin", 128, 512),
    }


RANDOM_PARAM_GRID = {
    "max_depth": [3, 4, 5, 6, 7, 8, 10],
    "learning_rate": np.linspace(0.015, 0.08, 10),
    "subsample": np.linspace(0.70, 1.00, 7),
    "colsample_bytree": np.linspace(0.70, 1.00, 7),
    "min_child_weight": [1, 2, 3, 5, 8, 12, 18],
    "reg_alpha": [0.0001, 0.001, 0.01, 0.1, 1.0, 5.0],
    "reg_lambda": [0.3, 0.7, 1.0, 2.0, 5.0, 10.0, 20.0],
    "gamma": [0, 0.05, 0.1, 0.3, 0.7, 1.5, 3.0],
    "max_bin": [128, 192, 256, 384, 512],
}


def tune_xgb(name, X_tr, y_tr, X_va, y_va, sample_weight=None, n_trials=25):
    if not TUNE_MODELS:
        params = {
            "max_depth": 6,
            "learning_rate": 0.035,
            "subsample": 0.90,
            "colsample_bytree": 0.90,
            "min_child_weight": 3,
            "reg_alpha": 0.01,
            "reg_lambda": 3.0,
            "gamma": 0.0,
            "max_bin": 256,
        }
        model = fit_model(make_model(params), X_tr, y_tr, X_va, y_va, sample_weight)
        pred = model.predict(X_va)
        return params, model, rmse(y_va, pred), best_rounds(model)

    print(f"Tuning {name} with {n_trials} trials...")

    best = {"score": np.inf, "params": None, "model": None}

    if HAS_OPTUNA:
        def objective(trial):
            params = optuna_params(trial)
            model = fit_model(make_model(params), X_tr, y_tr, X_va, y_va, sample_weight)
            pred = model.predict(X_va)
            score = rmse(y_va, pred)
            if score < best["score"]:
                best.update({"score": score, "params": params, "model": model})
            return score

        study = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))
        study.optimize(objective, n_trials=n_trials, show_progress_bar=False)
        params = study.best_params
        model = best["model"] if best["model"] is not None else fit_model(make_model(params), X_tr, y_tr, X_va, y_va, sample_weight)
        score = study.best_value
    else:
        sampler = ParameterSampler(RANDOM_PARAM_GRID, n_iter=n_trials, random_state=RANDOM_STATE)
        for i, params in enumerate(sampler, start=1):
            model = fit_model(make_model(params), X_tr, y_tr, X_va, y_va, sample_weight)
            pred = model.predict(X_va)
            score = rmse(y_va, pred)
            if score < best["score"]:
                best.update({"score": score, "params": params, "model": model})
            print(f"  {name} trial {i:02d}/{n_trials}: rmse={score:.4f}")
        params, model, score = best["params"], best["model"], best["score"]

    rounds = best_rounds(model)
    print(f"Best {name}: rmse={score:.4f}, rounds={rounds}, params={params}")
    return params, model, score, rounds


XGBoost training params: {'tree_method': 'hist', 'device': 'cuda'}


In [42]:
# =====================
# Train/tune global and specialist models
# =====================
global_params, global_model, global_score, global_rounds = tune_xgb(
    "global",
    X_train,
    y_train,
    X_valid,
    y_valid,
    sample_weight=global_weights,
    n_trials=N_TRIALS_GLOBAL,
)

weekday_mask_train = train_part_fe["is_weekend"].astype(int).to_numpy() == 0
weekend_mask_train = train_part_fe["is_weekend"].astype(int).to_numpy() == 1
weekday_mask_valid = valid_part_fe["is_weekend"].astype(int).to_numpy() == 0
weekend_mask_valid = valid_part_fe["is_weekend"].astype(int).to_numpy() == 1

y_weekday, weekday_weights, weekday_mad_info = mad_cap_target(
    train_part_fe.loc[weekday_mask_train].reset_index(drop=True),
    TARGET,
    "hour_of_week",
    MAD_Z_TARGET,
)
y_weekend, weekend_weights, weekend_mad_info = mad_cap_target(
    train_part_fe.loc[weekend_mask_train].reset_index(drop=True),
    TARGET,
    "hour_of_week",
    MAD_Z_TARGET,
)
print("Weekday MAD info:", weekday_mad_info)
print("Weekend MAD info:", weekend_mad_info)

weekday_params, weekday_model, weekday_score, weekday_rounds = tune_xgb(
    "weekday",
    X_train.loc[weekday_mask_train],
    y_weekday,
    X_valid.loc[weekday_mask_valid],
    y_valid[weekday_mask_valid],
    sample_weight=weekday_weights,
    n_trials=N_TRIALS_SPECIALIST,
)

weekend_params, weekend_model, weekend_score, weekend_rounds = tune_xgb(
    "weekend",
    X_train.loc[weekend_mask_train],
    y_weekend,
    X_valid.loc[weekend_mask_valid],
    y_valid[weekend_mask_valid],
    sample_weight=weekend_weights,
    n_trials=N_TRIALS_SPECIALIST,
)


Tuning global with 30 trials...
Best global: rmse=220.4327, rounds=2988, params={'max_depth': 3, 'learning_rate': 0.035864592100538176, 'subsample': 0.720477267323009, 'colsample_bytree': 0.7263931134040912, 'min_child_weight': 12.112946525917963, 'reg_alpha': 0.690571311043129, 'reg_lambda': 26.490004910326665, 'gamma': 2.4801201916002897, 'max_bin': 272}
Weekday MAD info: {'outlier_rate': 0.0, 'max_abs_robust_z': 3.220790918690602}
Weekend MAD info: {'outlier_rate': 0.0, 'max_abs_robust_z': 2.988960784313725}
Tuning weekday with 25 trials...
Best weekday: rmse=222.7509, rounds=1635, params={'max_depth': 4, 'learning_rate': 0.040093749681793335, 'subsample': 0.7141568745349266, 'colsample_bytree': 0.8865077992536115, 'min_child_weight': 5.4321453412350165, 'reg_alpha': 1.3625001376651005, 'reg_lambda': 9.757705155066942, 'gamma': 0.6449171304826775, 'max_bin': 266}
Tuning weekend with 25 trials...
Best weekend: rmse=236.4533, rounds=1602, params={'max_depth': 3, 'learning_rate': 0.044

In [43]:
# =====================
# Validation predictions and blend tuning
# =====================
valid_global_pred = global_model.predict(X_valid)

valid_specialist_pred = np.zeros(len(X_valid), dtype=float)
valid_specialist_pred[weekday_mask_valid] = weekday_model.predict(X_valid.loc[weekday_mask_valid])
valid_specialist_pred[weekend_mask_valid] = weekend_model.predict(X_valid.loc[weekend_mask_valid])

blend_rows = []
grid = np.linspace(0.0, 1.0, 41)  # weight on global model

best_blend = {"rmse": np.inf, "weekday_global_weight": None, "weekend_global_weight": None, "pred": None}
for wg_weekday in grid:
    for wg_weekend in grid:
        pred = np.zeros(len(X_valid), dtype=float)
        pred[weekday_mask_valid] = (
            wg_weekday * valid_global_pred[weekday_mask_valid]
            + (1 - wg_weekday) * valid_specialist_pred[weekday_mask_valid]
        )
        pred[weekend_mask_valid] = (
            wg_weekend * valid_global_pred[weekend_mask_valid]
            + (1 - wg_weekend) * valid_specialist_pred[weekend_mask_valid]
        )
        score = rmse(y_valid, pred)
        blend_rows.append({
            "weekday_global_weight": wg_weekday,
            "weekend_global_weight": wg_weekend,
            "rmse": score,
        })
        if score < best_blend["rmse"]:
            best_blend.update({
                "rmse": score,
                "weekday_global_weight": wg_weekday,
                "weekend_global_weight": wg_weekend,
                "pred": pred,
            })

print("Best blend:", {k: v for k, v in best_blend.items() if k != "pred"})

metrics = [
    metric_row("global", y_valid, valid_global_pred),
    metric_row("weekday_weekend_specialist", y_valid, valid_specialist_pred),
    metric_row("blended_global_specialist", y_valid, best_blend["pred"]),
]

metric_table = show_metrics(metrics)

blend_table = pd.DataFrame(blend_rows).sort_values("rmse").reset_index(drop=True)
display(blend_table.head(10))


Best blend: {'rmse': 218.56399067176116, 'weekday_global_weight': np.float64(0.5750000000000001), 'weekend_global_weight': np.float64(0.8250000000000001)}


,model,rmse,mae,mape_pct,smape_pct,r2
0,blended_global_specialist,218.563991,159.697175,1.795683,1.783664,0.968634
1,global,220.432698,160.890744,1.807384,1.795629,0.968096
2,weekday_weekend_specialist,226.751042,168.045264,1.899312,1.886722,0.966240


,weekday_global_weight,weekend_global_weight,rmse
0,0.575,0.825,218.563991
1,0.575,0.800,218.564986
2,0.600,0.825,218.565998
3,0.600,0.800,218.566993
4,0.575,0.850,218.572759
5,0.550,0.825,218.573573
6,0.550,0.800,218.574568
7,0.600,0.850,218.574766
8,0.575,0.775,218.575744
9,0.600,0.775,218.577751


In [44]:
# =====================
# Save validation diagnostics
# =====================
valid_diag = pd.DataFrame({
    DATE_COL: valid_part_fe[DATE_COL].values if DATE_COL in valid_part_fe.columns else np.arange(len(valid_part_fe)),
    "actual": y_valid,
    "pred_global": valid_global_pred,
    "pred_specialist": valid_specialist_pred,
    "pred_blended": best_blend["pred"],
    "is_weekend": valid_part_fe["is_weekend"].values,
})
valid_diag["abs_error_blended"] = np.abs(valid_diag["actual"] - valid_diag["pred_blended"])
valid_diag_path = OUTPUT_DIR / "validation_predictions.csv"
valid_diag.to_csv(valid_diag_path, index=False)
print("Saved:", valid_diag_path)
display(valid_diag.head())


Saved: /kaggle/working/validation_predictions.csv


,start_time,actual,pred_global,pred_specialist,pred_blended,is_weekend,abs_error_blended
0,2020-01-04 09:00:00,10154.667,10050.698242,10081.266602,10056.047705,1.0,98.619295
1,2020-01-04 10:00:00,10246.000,10086.508789,10121.893555,10092.701123,1.0,153.298877
2,2020-01-04 11:00:00,10313.000,10089.158203,10124.742188,10095.385400,1.0,217.614600
3,2020-01-04 12:00:00,10380.000,10126.066406,10185.278320,10136.428491,1.0,243.571509
4,2020-01-04 13:00:00,10447.000,10350.058594,10363.763672,10352.456982,1.0,94.543018


In [45]:
# =====================
# Final training on full train, then predict test
# =====================
full_base = add_base_features(train_raw)
test_base = add_base_features(test_raw)
if DATE_COL in full_base.columns:
    full_base = full_base.sort_values(DATE_COL).reset_index(drop=True)

# Target-stat features are fit on all training rows for final inference.
full_fe, test_fe = add_target_stat_features(full_base, [full_base, test_base], target=TARGET)

FEATURES_FINAL = get_feature_columns(full_fe)
missing_in_test = [c for c in FEATURES_FINAL if c not in test_fe.columns]
if missing_in_test:
    raise ValueError(f"Missing feature columns in test: {missing_in_test}")

X_full = full_fe[FEATURES_FINAL].replace([np.inf, -np.inf], np.nan)
X_test = test_fe[FEATURES_FINAL].replace([np.inf, -np.inf], np.nan)

if USE_MAD_FEATURE_CLIP:
    final_clipper = MADFeatureClipper(MAD_Z_FEATURES).fit(X_full)
    X_full = final_clipper.transform(X_full)
    X_test = final_clipper.transform(X_test)

y_full_global, full_global_weights, _ = mad_cap_target(full_fe, TARGET, "hour_of_week", MAD_Z_TARGET)

final_global = fit_model(
    make_model(global_params, n_estimators=global_rounds, early_stopping=False),
    X_full,
    y_full_global,
    sample_weight=full_global_weights,
)

full_weekday_mask = full_fe["is_weekend"].astype(int).to_numpy() == 0
full_weekend_mask = full_fe["is_weekend"].astype(int).to_numpy() == 1
test_weekday_mask = test_fe["is_weekend"].astype(int).to_numpy() == 0
test_weekend_mask = test_fe["is_weekend"].astype(int).to_numpy() == 1

y_full_weekday, full_weekday_weights, _ = mad_cap_target(
    full_fe.loc[full_weekday_mask].reset_index(drop=True),
    TARGET,
    "hour_of_week",
    MAD_Z_TARGET,
)
y_full_weekend, full_weekend_weights, _ = mad_cap_target(
    full_fe.loc[full_weekend_mask].reset_index(drop=True),
    TARGET,
    "hour_of_week",
    MAD_Z_TARGET,
)

final_weekday = fit_model(
    make_model(weekday_params, n_estimators=weekday_rounds, early_stopping=False),
    X_full.loc[full_weekday_mask],
    y_full_weekday,
    sample_weight=full_weekday_weights,
)

final_weekend = fit_model(
    make_model(weekend_params, n_estimators=weekend_rounds, early_stopping=False),
    X_full.loc[full_weekend_mask],
    y_full_weekend,
    sample_weight=full_weekend_weights,
)

test_global_pred = final_global.predict(X_test)
test_specialist_pred = np.zeros(len(X_test), dtype=float)
test_specialist_pred[test_weekday_mask] = final_weekday.predict(X_test.loc[test_weekday_mask])
test_specialist_pred[test_weekend_mask] = final_weekend.predict(X_test.loc[test_weekend_mask])

test_pred = np.zeros(len(X_test), dtype=float)
wg_weekday = best_blend["weekday_global_weight"]
wg_weekend = best_blend["weekend_global_weight"]
test_pred[test_weekday_mask] = (
    wg_weekday * test_global_pred[test_weekday_mask]
    + (1 - wg_weekday) * test_specialist_pred[test_weekday_mask]
)
test_pred[test_weekend_mask] = (
    wg_weekend * test_global_pred[test_weekend_mask]
    + (1 - wg_weekend) * test_specialist_pred[test_weekend_mask]
)

test_pred = np.maximum(test_pred, 0)
print("Prediction summary:")
display(pd.Series(test_pred).describe())


Prediction summary:


count     8757.000000
mean      9821.942349
std       1583.330102
min       6830.618359
25%       8552.240454
50%       9454.306714
75%      11000.786890
max      14004.232031
dtype: float64

In [46]:
# =====================
# Optional local test metrics if target exists in test_clean.csv
# Kaggle hidden test normally will not include the true target.
# =====================
if TARGET in test_raw.columns and test_raw[TARGET].notna().any():
    y_test_available = pd.to_numeric(test_raw[TARGET], errors="coerce").to_numpy()
    ok = np.isfinite(y_test_available)
    if ok.any():
        test_metrics = show_metrics([
            metric_row("test_global", y_test_available[ok], test_global_pred[ok]),
            metric_row("test_specialist", y_test_available[ok], test_specialist_pred[ok]),
            metric_row("test_blended", y_test_available[ok], test_pred[ok]),
        ])


,model,rmse,mae,mape_pct,smape_pct,r2
0,test_blended,219.699894,162.292505,1.663740,1.664436,0.981198
1,test_global,222.526894,164.947065,1.691704,1.692621,0.980711
2,test_specialist,223.763874,164.604921,1.683685,1.684583,0.980496


## Leakage Audit and Feature Contribution

`ewm_24h`, `rolling_mean_24h`, and other rolling features are expected to be powerful for energy forecasting. They are safe only if they were created from past values, for example with `consumption.shift(1)` before rolling/EWM. This section checks correlation, feature gain importance, and an ablation model that removes the strongest leakage-suspect rolling features.


In [47]:
# =====================
# Leakage audit and feature contribution
# =====================
LEAKAGE_SUSPECT_FEATURES = [
    "ewm_24h",
    "rolling_mean_24h",
    "rolling_mean_168h",
    "rolling_std_24h",
]

audit_cols = [
    TARGET,
    "ewm_24h",
    "rolling_mean_24h",
    "rolling_mean_168h",
    "rolling_std_24h",
    "lag_24h",
    "lag_168h",
]
audit_cols = [c for c in audit_cols if c in train_raw.columns]

print("Correlation with target. Very high values are not automatically leakage, but they should be checked.")
corr_with_target = (
    train_raw[audit_cols]
    .corr(numeric_only=True)[TARGET]
    .sort_values(ascending=False)
)
display(corr_with_target)

inspect_cols = [DATE_COL, TARGET, "ewm_24h", "rolling_mean_24h", "lag_24h", "lag_168h"]
inspect_cols = [c for c in inspect_cols if c in train_raw.columns]
display(train_raw[inspect_cols].head(30))


def xgb_gain_importance(model, feature_names, model_name):
    booster = model.get_booster()
    score = booster.get_score(importance_type="gain")

    rows = []
    for i, feature in enumerate(feature_names):
        gain = score.get(feature, score.get(f"f{i}", 0))
        rows.append({
            "model": model_name,
            "feature": feature,
            "gain_importance": gain,
        })

    out = pd.DataFrame(rows)
    total = out["gain_importance"].sum()
    out["importance_pct"] = 100 * out["gain_importance"] / total if total > 0 else 0
    return out.sort_values("gain_importance", ascending=False).reset_index(drop=True)


global_imp = xgb_gain_importance(global_model, FEATURES, "global")
weekday_imp = xgb_gain_importance(weekday_model, FEATURES, "weekday")
weekend_imp = xgb_gain_importance(weekend_model, FEATURES, "weekend")

importance_table = pd.concat([global_imp, weekday_imp, weekend_imp], ignore_index=True)
combined_importance = (
    importance_table
    .groupby("feature", as_index=False)["gain_importance"]
    .mean()
    .sort_values("gain_importance", ascending=False)
    .reset_index(drop=True)
)
combined_total = combined_importance["gain_importance"].sum()
combined_importance["importance_pct"] = 100 * combined_importance["gain_importance"] / combined_total

print("Top combined feature contribution across global, weekday, and weekend models:")
display(combined_importance.head(30))

print("Top global model features:")
display(global_imp.head(20))

features_no_suspect = [c for c in FEATURES if c not in LEAKAGE_SUSPECT_FEATURES]
removed_features = [c for c in FEATURES if c in LEAKAGE_SUSPECT_FEATURES]
print("Removed for ablation:", removed_features)

model_no_suspect = fit_model(
    make_model(global_params),
    X_train[features_no_suspect],
    y_train,
    X_valid[features_no_suspect],
    y_valid,
    sample_weight=global_weights,
)
pred_no_suspect = model_no_suspect.predict(X_valid[features_no_suspect])

ablation_metrics = pd.DataFrame([
    metric_row("global_with_all_features", y_valid, valid_global_pred),
    metric_row("global_without_suspect_features", y_valid, pred_no_suspect),
])
display(ablation_metrics)


Correlation with target. Very high values are not automatically leakage, but they should be checked.


consumption          1.000000
ewm_24h              0.956161
rolling_mean_24h     0.950084
lag_24h              0.917134
rolling_mean_168h    0.900875
lag_168h             0.897324
rolling_std_24h      0.166418
Name: consumption, dtype: float64

,start_time,consumption,ewm_24h,rolling_mean_24h,lag_24h,lag_168h
0,2016-01-14 21:00:00,12591.0,13222.077680,13202.666667,12597.0,14074.0
1,2016-01-14 22:00:00,13155.0,13171.591466,13202.416667,13225.0,13643.0
2,2016-01-14 23:00:00,13216.5,13170.264149,13199.500000,13193.0,14639.0
3,2016-01-15 00:00:00,13290.5,13173.963017,13200.479167,13178.0,14592.0
4,2016-01-15 01:00:00,13290.5,13183.285976,13205.166667,13150.0,14483.5
5,2016-01-15 02:00:00,13264.5,13191.863097,13211.020833,13150.0,14394.0
6,2016-01-15 03:00:00,13264.5,13197.674050,13215.791667,13150.0,13602.0
7,2016-01-15 04:00:00,13110.0,13203.020126,13220.562500,12967.0,14414.0
8,2016-01-15 05:00:00,13596.0,13195.578516,13226.520833,13393.0,14631.0
9,2016-01-15 06:00:00,13718.0,13227.612234,13234.979167,13528.0,14802.0


Top combined feature contribution across global, weekday, and weekend models:


,feature,gain_importance,importance_pct
0,ewm_24h,3.787511e+08,49.274001
1,rolling_mean_24h,1.827715e+08,23.777838
2,lag_168h,4.012906e+07,5.220630
3,lag_24h,3.088920e+07,4.018561
4,weekend_hour_target_mean,1.843724e+07,2.398610
5,weekend_hour_target_median,1.226433e+07,1.595540
6,lag_mean_24_72,1.184777e+07,1.541347
7,month_target_mean,1.018565e+07,1.325113
8,prev_daily_max,9.186838e+06,1.195171
9,how_target_mean,7.804792e+06,1.015372


Top global model features:


,model,feature,gain_importance,importance_pct
0,global,ewm_24h,5.226136e+08,42.829805
1,global,rolling_mean_24h,4.226542e+08,34.637826
2,global,lag_168h,8.756133e+07,7.175922
3,global,weekend_hour_target_mean,2.722244e+07,2.230964
4,global,weekend_hour_target_median,2.208398e+07,1.809850
5,global,how_target_mean,1.618080e+07,1.326067
6,global,month_cos,1.489541e+07,1.220725
7,global,hour_target_mean,9.154004e+06,0.750199
8,global,lag_24h,6.100789e+06,0.499979
9,global,hour_of_week_cos,5.918439e+06,0.485034


Removed for ablation: ['rolling_mean_24h', 'rolling_mean_168h', 'rolling_std_24h', 'ewm_24h']


,model,rmse,mae,mape_pct,smape_pct,r2
0,global_with_all_features,220.432698,160.890744,1.807384,1.795629,0.968096
1,global_without_suspect_features,262.242769,195.856800,2.204349,2.178978,0.954845


In [48]:
# =====================
# Create Kaggle submission
# =====================
def find_sample_submission():
    root = Path("/kaggle/input")
    if not root.exists():
        return None
    matches = sorted(root.rglob("sample_submission*.csv"))
    return matches[0] if matches else None


sample_path = find_sample_submission()
if sample_path is not None:
    submission = pd.read_csv(sample_path)
    candidate_target_cols = [c for c in submission.columns if c.lower() in [TARGET.lower(), "target", "prediction", "pred"]]
    if candidate_target_cols:
        pred_col = candidate_target_cols[0]
    else:
        pred_col = submission.columns[-1]
    submission[pred_col] = test_pred
else:
    if DATE_COL in test_raw.columns:
        submission = pd.DataFrame({DATE_COL: test_raw[DATE_COL], TARGET: test_pred})
    elif "id" in test_raw.columns:
        submission = pd.DataFrame({"id": test_raw["id"], TARGET: test_pred})
    else:
        submission = pd.DataFrame({"id": np.arange(len(test_pred)), TARGET: test_pred})

submission_path = OUTPUT_DIR / "submission.csv"
submission.to_csv(submission_path, index=False)
print("Saved submission:", submission_path)
display(submission.head())


Saved submission: /kaggle/working/submission.csv


,start_time,consumption
0,2021-01-01 00:00:00,10153.516626
1,2021-01-01 01:00:00,10105.064209
2,2021-01-01 02:00:00,9896.844775
3,2021-01-01 03:00:00,9792.389209
4,2021-01-01 04:00:00,9485.815112


In [49]:
# =====================
# Robustness checks
# =====================
import numpy as np
import pandas as pd

ROBUSTNESS_SEED = 42
rng = np.random.default_rng(ROBUSTNESS_SEED)


def predict_blended_from_X(X, meta_df):
    """Predict using global + correct weekday/weekend specialist."""
    global_pred = global_model.predict(X)

    is_weekend_arr = meta_df["is_weekend"].astype(int).to_numpy() == 1
    specialist_pred = np.zeros(len(X), dtype=float)

    specialist_pred[~is_weekend_arr] = weekday_model.predict(X.loc[~is_weekend_arr])
    specialist_pred[is_weekend_arr] = weekend_model.predict(X.loc[is_weekend_arr])

    pred = np.zeros(len(X), dtype=float)

    wg_weekday = best_blend["weekday_global_weight"]
    wg_weekend = best_blend["weekend_global_weight"]

    pred[~is_weekend_arr] = (
        wg_weekday * global_pred[~is_weekend_arr]
        + (1 - wg_weekday) * specialist_pred[~is_weekend_arr]
    )

    pred[is_weekend_arr] = (
        wg_weekend * global_pred[is_weekend_arr]
        + (1 - wg_weekend) * specialist_pred[is_weekend_arr]
    )

    return np.maximum(pred, 0)


def metrics_dict(name, y_true, y_pred):
    row = metric_row(name, y_true, y_pred)
    return row


# 1. Overall validation score
base_pred = predict_blended_from_X(X_valid, valid_part_fe)

robustness_rows = [
    metrics_dict("base_validation", y_valid, base_pred)
]


# 2. Slice robustness: weekday/weekend, night, peaks, holiday, month
slice_specs = {
    "weekday_only": valid_part_fe["is_weekend"].astype(int).to_numpy() == 0,
    "weekend_only": valid_part_fe["is_weekend"].astype(int).to_numpy() == 1,
    "night_hours": valid_part_fe["hour"].isin([0, 1, 2, 3, 4, 5]).to_numpy(),
    "morning_peak": valid_part_fe["hour"].between(7, 10).to_numpy(),
    "evening_peak": valid_part_fe["hour"].between(17, 22).to_numpy(),
}

if "is_holiday" in valid_part_fe.columns:
    slice_specs["holiday_only"] = valid_part_fe["is_holiday"].astype(int).to_numpy() == 1
    slice_specs["non_holiday"] = valid_part_fe["is_holiday"].astype(int).to_numpy() == 0

for name, mask in slice_specs.items():
    if mask.sum() >= 20:
        robustness_rows.append(
            metrics_dict(
                f"slice_{name}",
                y_valid[mask],
                base_pred[mask],
            )
        )

# Monthly stability
if "month" in valid_part_fe.columns:
    for month_value in sorted(valid_part_fe["month"].dropna().unique()):
        mask = valid_part_fe["month"].to_numpy() == month_value
        if mask.sum() >= 50:
            robustness_rows.append(
                metrics_dict(
                    f"month_{int(month_value)}",
                    y_valid[mask],
                    base_pred[mask],
                )
            )


# 3. Bootstrap confidence interval for RMSE and MAE
boot_rows = []
n = len(y_valid)

for i in range(300):
    idx = rng.integers(0, n, size=n)
    boot_rows.append({
        "rmse": rmse(y_valid[idx], base_pred[idx]),
        "mae": mean_absolute_error(y_valid[idx], base_pred[idx]),
        "mape_pct": mape(y_valid[idx], base_pred[idx]),
    })

boot_df = pd.DataFrame(boot_rows)

bootstrap_summary = boot_df.quantile([0.025, 0.50, 0.975]).T
bootstrap_summary.columns = ["lower_2_5_pct", "median", "upper_97_5_pct"]

print("Bootstrap confidence intervals:")
display(bootstrap_summary)


# 4. Input noise stress test
# This checks whether small noise in lag/rolling features destroys performance.
stress_features = [
    "ewm_24h",
    "rolling_mean_24h",
    "rolling_mean_168h",
    "lag_24h",
    "lag_48h",
    "lag_72h",
    "lag_168h",
    "prev_daily_mean",
    "prev_daily_max",
]
stress_features = [c for c in stress_features if c in X_valid.columns]

for noise_level in [0.01, 0.025, 0.05, 0.10]:
    X_noisy = X_valid.copy()

    for col in stress_features:
        col_std = X_train[col].std()
        if pd.notna(col_std) and col_std > 0:
            X_noisy[col] = X_noisy[col] + rng.normal(
                loc=0,
                scale=noise_level * col_std,
                size=len(X_noisy),
            )

    noisy_pred = predict_blended_from_X(X_noisy, valid_part_fe)

    robustness_rows.append(
        metrics_dict(
            f"noise_{int(noise_level * 100)}pct_on_lag_rolling",
            y_valid,
            noisy_pred,
        )
    )


# 5. Outlier stress test
# This injects extreme lag/rolling values into random validation rows.
for outlier_rate in [0.01, 0.03, 0.05]:
    X_outlier = X_valid.copy()
    outlier_idx = rng.choice(
        np.arange(len(X_outlier)),
        size=max(1, int(len(X_outlier) * outlier_rate)),
        replace=False,
    )

    for col in stress_features:
        col_std = X_train[col].std()
        if pd.notna(col_std) and col_std > 0:
            direction = rng.choice([-1, 1], size=len(outlier_idx))
            X_outlier.loc[X_outlier.index[outlier_idx], col] = (
                X_outlier.loc[X_outlier.index[outlier_idx], col]
                + direction * 5 * col_std
            )

    outlier_pred = predict_blended_from_X(X_outlier, valid_part_fe)

    robustness_rows.append(
        metrics_dict(
            f"outlier_injection_{int(outlier_rate * 100)}pct_rows",
            y_valid,
            outlier_pred,
        )
    )


# 6. Feature dependence stress test
# Replaces important features with train median one-by-one.
important_features_to_test = [
    "ewm_24h",
    "rolling_mean_24h",
    "lag_168h",
    "lag_24h",
    "weekend_hour_target_mean",
    "month_target_mean",
    "prev_daily_max",
]
important_features_to_test = [c for c in important_features_to_test if c in X_valid.columns]

for col in important_features_to_test:
    X_removed = X_valid.copy()
    X_removed[col] = X_train[col].median()

    removed_pred = predict_blended_from_X(X_removed, valid_part_fe)

    robustness_rows.append(
        metrics_dict(
            f"median_replace_{col}",
            y_valid,
            removed_pred,
        )
    )


robustness_table = pd.DataFrame(robustness_rows)
robustness_table = robustness_table.sort_values("rmse").reset_index(drop=True)

print("Robustness table:")
display(robustness_table)


Bootstrap confidence intervals:


,lower_2_5_pct,median,upper_97_5_pct
rmse,213.723789,218.962826,223.250709
mae,156.809714,159.756699,163.066691
mape_pct,1.760947,1.797431,1.832745


Robustness table:


,model,rmse,mae,mape_pct,smape_pct,r2
0,slice_morning_peak,144.180952,111.419629,1.205379,1.203459,0.985211
1,month_7,172.785831,122.449158,1.615472,1.606397,0.833609
2,month_5,185.114444,140.578990,1.741438,1.731458,0.905794
3,month_4,187.092730,142.838261,1.559446,1.554630,0.886566
4,month_8,194.204469,140.452224,1.824594,1.815013,0.847878
5,month_6,200.088842,141.348917,1.906493,1.889157,0.900281
6,month_9,216.480590,154.963412,1.938557,1.915057,0.860591
7,slice_weekday_only,218.377040,157.465964,1.728551,1.717070,0.965989
8,noise_1pct_on_lag_rolling,218.456580,159.563948,1.794006,1.781950,0.968665
9,base_validation,218.563991,159.697175,1.795683,1.783664,0.968634


## Final Notes

For a fast first run, keep trials around 10-15. For a serious Kaggle submission, increase trials to 50+ for the global model and 40+ for specialists. The validation score to trust is the blended model score from the time-based holdout, not the local test score if your test file happens to contain `consumption`.
